# Tune the star selection and the match filter on a known stream

As `gc_dwarf_optimization_example.ipynb` on a dwarf galaxy, on a galstreams stream crossing the footprint: its on
region along the track and its off regions on both sides (`analysis.stream_optimization`), the map and the Hess
diagrams, the SNR of its excess, and the SNR for every combination of star / galaxy classification, `POINT_LIKE_PROB`
cut, isochrone model and match filter width (as `population_optimization.ipynb`).

The regions follow the track in phi1 slices: in each, the track is at its galstreams phi2 and distance modulus; the
match filter of a star is at the distance modulus of its slice. Widths in units of the stream width (galstreams
`width_phi2` by default). The catalog magnitudes are already dereddened.

In [ ]:
%load_ext autoreload
%autoreload 2

import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from streameucliddes.analysis.catalog_preparation import catalog_match_filter_mask, prepare_match_filter_catalog
from streameucliddes.analysis.cmd_match_filter import isochrone_cmd_track, match_filter_cmd_columns, on_off_hess
from streameucliddes.analysis.gc_dwarf_optimization import compare_selections, selection_snr
from streameucliddes.analysis.stream_optimization import (
    stream_field,
    stream_object_healpix_mask,
    stream_regions,
    track_match_filter_mask,
)
from streameucliddes.catalogs.euclid_catalog import catalog_point_map, default_star_cuts, get_euclid_stars
from streameucliddes.catalogs.galstreams_catalog import (
    galstreams_track_in_own_frame,
    galstreams_width_phi2,
    load_galstreams,
    select_galstreams_in_footprint,
)
from streameucliddes.catalogs.isochrone_parameters import default_isochrone_grid, known_stream_isochrone_parameters
from streameucliddes.catalogs.literature_tracks import li2021_aliqa_uma_track, li2021_aliqa_uma_width_deg
from streameucliddes.plotting.plot_cmd import plot_hess, plot_isochrone, plot_match_filter_polygon
from streameucliddes.plotting.plot_gc_dwarf_optimization import plot_object_hess, plot_selection_comparison
from streameucliddes.plotting.plot_stream_optimization import plot_stream_map
from streameucliddes.plotting.plot_style import escape_text

## 1. Streams crossing the footprint

In [ ]:
CATALOG = "euclid_dr1"

# Footprint from the HEALPix point map of the catalog (no star read): True outside
footprint = catalog_point_map(CATALOG) == 0
# Every galstreams track (True: the non-default ones too, e.g. ATLAS-I21 and ATLAS-I24 of Ibata et al. 2021 and
# 2024), or the default one of each stream (False). The Ibata et al. 2024 tracks have no distance: only listed
# with add_ibata2024, and stream_regions needs a distance
ALL_TRACKS = True
mws = load_galstreams(all_tracks=ALL_TRACKS)
selection = select_galstreams_in_footprint(mws, footprint, nest=True, add_ibata2024=True)
selection

## 2. One stream: parameters, stars and regions

`SIGMA_DEG` None: the galstreams width of the track. `ISOCHRONE_SOURCE`: `"default"`, the fixed isochrone of a
generic search (`default_isochrone_grid`), or `"known"`, that of the stream (`stream_populations.csv`), in each
isochrone model. `TRACK_SOURCE`: the galstreams track `TRACK_NAME` (`OWN_FRAME`: in the great-circle frame
through its ends), or a fitted track of the literature that galstreams does not have (`catalogs.literature_tracks`).

In [ ]:
# Track: "galstreams" (TRACK_NAME), or "li2021_aliqa_uma": the fitted spline track of Aliqa Uma of Li et al. 2021
# (their DES density fit; galstreams AAU-AliqaUma-L21 is their approximate Eq. 3, 0.5 to 0.9 deg below it), with
# their width and distance (Eq. 2), in the frame through its ends
TRACK_SOURCE = "li2021_aliqa_uma"
TRACK_NAME = "AAU-AliqaUma-L21"
# The track in the great-circle frame through its own ends (True), or in its galstreams frame (False), which
# galstreams can share between tracks of one stream (AAU-ATLAS-L21 and AAU-AliqaUma-L21: Aliqa Uma tilted and
# kinked in the ATLAS frame)
OWN_FRAME = True
# Shift of the regions from the galstreams track in phi2 (deg), per track, set by hand where the stream is not on its
# track in DR1; the other tracks: none. Aliqa Uma: its match filter excess peaks about +0.6 deg from the Li 2021
# track (in its own frame)
PHI2_OFFSETS_DEG = {"AAU-AliqaUma-L21": 0.0}
SIGMA_DEG = None  # width of the stream in phi2 (deg); None: that of the track (galstreams width_phi2, Li 2021)
ON_N_SIGMA = 1.4  # half-width of the on region, in widths
OFF_N_SIGMA = (3.5, 6.0)  # off regions on both sides, in widths
SLICE_DEG = 0.5  # phi1 slices of the regions (track phi2 and distance per slice)
PHI1_RANGE = None  # None: the track in the footprint
MASKED_OBJECT_TYPES = ("dwarfs", "gc", "galaxies")  # known objects masked, as in the searches

# Star / galaxy classifications, read with the loosest POINT_LIKE_PROB threshold so that POINT_LIKE_PROB_CUTS can be
# applied on top
MIN_POINT_LIKE_PROB = 0.05
CLASSIFICATIONS = {
    "morphology": default_star_cuts(CATALOG, "morphology_cuts", min_point_like_prob=MIN_POINT_LIKE_PROB),
    "gatto": default_star_cuts(CATALOG, "gatto_stars_dr1"),  # POINT_LIKE_PROB >= 0.05 and the colour box
}
POINT_LIKE_PROB_CUTS = np.round(np.arange(MIN_POINT_LIKE_PROB, 0.951, 0.05), 2)  # stars kept: POINT_LIKE_PROB >= cut

ISOCHRONE_SOURCE = "default"  # "default" (generic search) or "known" (stream_populations.csv)
ISOCHRONE_MODELS = ("Marigo2017", "Dotter2016")
MATCH_FILTER = {"name": "vis_h", "reference_band": ("euclid_dr1", "VIS"), "color_bands": [("euclid_dr1", "H")]}
# Width of the match filter: error_multiplier times the photometric error plus color_spread, each (blue side, red side);
# absent: the streamobs defaults, (2, 2) and (0.05, 0.05) mag
WIDTHS = {
    "default": {},
    "color_spread (0.05, 0.1)": {"color_spread": (0.05, 0.1)},  # DES 2018 (Shipp et al.)
    "color_spread (0.1, 0.1)": {"color_spread": (0.1, 0.1)},
    "color_spread (0.1, 0.15)": {"color_spread": (0.1, 0.15)},
    "error_multiplier (1, 1)": {"error_multiplier": (1.0, 1.0)},
    "error_multiplier (3, 3)": {"error_multiplier": (3.0, 3.0)},
    "color_spread (0.1, 0.1) + error_multiplier (3, 3)": {"color_spread": (0.1, 0.1), "error_multiplier": (3.0, 3.0)},
}
# Reference scenario of the relative SNR: the default selection of the searches
REFERENCE = {"classification": "morphology", "isochrone_model": "Marigo2017", "width": "default", "point_like_prob_cut": 0.5}

# Plot styles
CLASSIFICATION_COLORS = {"morphology": "C0", "gatto": "C1"}
ISOCHRONE_MODEL_COLORS = {"Marigo2017": "k", "Dotter2016": "C2"}  # Hess diagrams
ISOCHRONE_MODEL_MARKERS = {"Marigo2017": "o", "Dotter2016": "^"}
ISOCHRONE_MODEL_LINESTYLES = {"Marigo2017": "-", "Dotter2016": ":"}

if TRACK_SOURCE == "li2021_aliqa_uma":
    track = li2021_aliqa_uma_track()
    sigma_deg = li2021_aliqa_uma_width_deg() if SIGMA_DEG is None else SIGMA_DEG
else:
    track = galstreams_track_in_own_frame(mws[TRACK_NAME]) if OWN_FRAME else mws[TRACK_NAME]
    sigma_deg = galstreams_width_phi2(TRACK_NAME, mws) if SIGMA_DEG is None else SIGMA_DEG
stream_name = track.stream_name
if ISOCHRONE_SOURCE == "default":
    ISOCHRONES = {model: default_isochrone_grid(isochrone_model=model)[0] for model in ISOCHRONE_MODELS}
else:
    ISOCHRONES = {model: known_stream_isochrone_parameters(stream_name, isochrone_model=model)[0]
                  for model in ISOCHRONE_MODELS}
phi2_offset_deg = PHI2_OFFSETS_DEG.get(track.track_name, 0.0)
field = stream_field(track, sigma_deg=sigma_deg, off_n_sigma=OFF_N_SIGMA, footprint_mask=footprint, nest=True,
                     phi1_range=PHI1_RANGE, phi2_offset_deg=phi2_offset_deg)
print(f"{track.track_name} ({stream_name}): width {sigma_deg:.3f} deg, regions {phi2_offset_deg:+g} deg from the track, phi1 {field['phi1_range'][0]:.1f} to "
      f"{field['phi1_range'][1]:.1f} deg; box ra {field['ra_range']}, dec {field['dec_range']}")
print("isochrones:", ISOCHRONES)

In [ ]:
# Stars and regions of each classification (the object mask once)
object_mask = stream_object_healpix_mask(MASKED_OBJECT_TYPES)
fields = {}  # classification: (stars, regions)
for classification, cuts in CLASSIFICATIONS.items():
    stars = get_euclid_stars(CATALOG, cuts=cuts, ra_range=field["ra_range"], dec_range=field["dec_range"], verbose=False)
    regions = stream_regions(stars, track, phi1_range=field["phi1_range"], sigma_deg=sigma_deg, on_n_sigma=ON_N_SIGMA,
                             off_n_sigma=OFF_N_SIGMA, slice_deg=SLICE_DEG, object_healpix_mask=object_mask,
                             phi2_offset_deg=phi2_offset_deg)
    fields[classification] = (stars, regions)
    print(f"{classification}: {len(stars):,} stars; on {regions['on'].sum():,} ({regions['on_area_deg2']:.2f} deg2, "
          f"{regions['on_area_fraction']:.0%} kept), off {regions['off'].sum():,} ({regions['off_area_deg2']:.2f} deg2, "
          f"{regions['off_area_fraction']:.0%} kept)")
print("distance moduli of the slices:", fields["morphology"][1]["slice_dm"])

## 3. Map of the match filter stars, with the regions

In [ ]:
MAP_CLASSIFICATION, MAP_POINT_LIKE_PROB_CUT, MAP_ISOCHRONE_MODEL = "morphology", 0.5, "Marigo2017"
MAP_SMOOTHING_DEG = 0.05

stars, regions = fields[MAP_CLASSIFICATION]
map_selection = (track_match_filter_mask(stars, regions, ISOCHRONES[MAP_ISOCHRONE_MODEL], MATCH_FILTER)
                 & (stars["POINT_LIKE_PROB"].to_numpy() >= MAP_POINT_LIKE_PROB_CUT))
fig, ax = plot_stream_map(regions, map_selection, smoothing_deg=MAP_SMOOTHING_DEG)  # white: masked (footprint, objects)

Maps with the match filter at one distance modulus over the whole map (not that of the track), from `DM_MAPS`:
where and at which distance the stream stands out, against the regions (offset `phi2_offset_deg` from the
galstreams track, dashed). Title: SNR of the regions for that map.

In [ ]:
DM_MAPS = np.round(np.arange(16.0, 17.51, 0.25), 2)
DM_MAP_SCENARIO = {"classification": "morphology", "point_like_prob_cut": 0.5, "isochrone_model": "Marigo2017",
                   "width": "default"}
DM_MAP_SMOOTHING_DEG = 0.05

stars, regions = fields[DM_MAP_SCENARIO["classification"]]
kept = stars["POINT_LIKE_PROB"].to_numpy() >= DM_MAP_SCENARIO["point_like_prob_cut"]
isochrone = ISOCHRONES[DM_MAP_SCENARIO["isochrone_model"]]
match_filter = {**MATCH_FILTER, **WIDTHS[DM_MAP_SCENARIO["width"]]}
fig, axes = plt.subplots(len(DM_MAPS), 1, figsize=(11.0, 3.6 * len(DM_MAPS)), layout="constrained")
for ax, dm in zip(np.atleast_1d(axes), DM_MAPS):
    selected = catalog_match_filter_mask(stars, dm, isochrone, match_filter) & kept
    snr = selection_snr(regions, selected)
    plot_stream_map(regions, selected, smoothing_deg=DM_MAP_SMOOTHING_DEG, ax=ax,
                    title=f"dm {dm:g}: {snr['n_signal']:.0f} $\\pm$ {snr['n_signal_error']:.0f} excess stars, "
                          f"SNR {snr['snr']:.1f}")
fig.suptitle(escape_text(f"{track.track_name}: {DM_MAP_SCENARIO['classification']}, POINT_LIKE_PROB >= "
                         f"{DM_MAP_SCENARIO['point_like_prob_cut']}, {DM_MAP_SCENARIO['isochrone_model']}, "
                         f"width {DM_MAP_SCENARIO['width']}"))
plt.show()

## 4. Hess diagrams

On, off and on - off of the stars passing the error cuts of the match filter (one row per colour); then the on - off of
each classification, with the isochrone and match filter of each model at the mean distance modulus of the track,
and the SNR of each match filter (at the distance of the track along it).

In [ ]:
stars, regions = fields["morphology"]
mean_dm = float(np.mean(regions["slice_dm"]))
shown = prepare_match_filter_catalog(stars, MATCH_FILTER, as_mask=True)  # its error cuts
fig, axes = plot_object_hess(stars, regions, mean_dm, ISOCHRONES["Marigo2017"], MATCH_FILTER, stars=shown,
                             mag_bins=np.arange(18.0, 26.01, 1 / 6))

In [ ]:
mag_cols = match_filter_cmd_columns(MATCH_FILTER)[0]  # (VIS, H)
HESS_POINT_LIKE_PROB_CUTS = {"morphology": 0.5, "gatto": 0.05}

fig, axes = plt.subplots(1, len(fields), figsize=(6 * len(fields), 5), sharey=True, layout="constrained")
for ax, (classification, (stars, regions)) in zip(np.atleast_1d(axes), fields.items()):
    kept = stars["POINT_LIKE_PROB"].to_numpy() >= HESS_POINT_LIKE_PROB_CUTS[classification]
    shown = prepare_match_filter_catalog(stars, MATCH_FILTER, as_mask=True) & kept
    _, _, difference = on_off_hess(stars, mag_cols, shown & regions["on"], regions["on_area_deg2"],
                                   [shown & regions["off"]], [regions["off_area_deg2"]],
                                   mag_bins=np.arange(18.0, 26.01, 1 / 6))
    plot_hess(difference, ax=ax, difference=True, smoothing_bins=1,
              title=f"{classification} ({escape_text('POINT_LIKE_PROB')} $\\geq$ {HESS_POINT_LIKE_PROB_CUTS[classification]}): "
                    "on $-$ off, "
                    f"{difference['n_stars']:.0f} excess stars")
    for row, (isochrone_model, color) in enumerate(ISOCHRONE_MODEL_COLORS.items()):
        isochrone = ISOCHRONES[isochrone_model]
        track_cmd = isochrone_cmd_track(mean_dm, isochrone["age"], isochrone["metallicity"], match_filter=MATCH_FILTER,
                                        isochrone_model=isochrone_model)
        plot_isochrone(ax, track_cmd, color=color, label=isochrone_model)
        plot_match_filter_polygon(ax, mean_dm, isochrone, MATCH_FILTER, color=color, ls=":",
                                  label=f"{isochrone_model} match filter")
        snr = selection_snr(regions, track_match_filter_mask(stars, regions, isochrone, MATCH_FILTER) & kept)["snr"]
        ax.text(0.97, 0.97 - 0.06 * row, f"SNR = {snr:.1f}", transform=ax.transAxes, ha="right", va="top", color=color)
    ax.legend(loc="upper left", fontsize="small")
fig.suptitle(escape_text(f"{track.track_name}, dm {mean_dm:.2f} (mean of the track)"))
plt.show()

## 5. SNR of every scenario

One row per classification, isochrone model, width and `POINT_LIKE_PROB` cut. `relative_snr`: over the SNR of
`REFERENCE`.

In [ ]:
rows = []
for classification, (stars, regions) in fields.items():
    point_like_prob = stars["POINT_LIKE_PROB"].to_numpy()
    for (isochrone_model, isochrone), (width_name, width) in itertools.product(ISOCHRONES.items(), WIDTHS.items()):
        in_match_filter = track_match_filter_mask(stars, regions, isochrone, {**MATCH_FILTER, **width})
        for cut in POINT_LIKE_PROB_CUTS:
            rows.append({"classification": classification, "isochrone_model": isochrone_model, "width": width_name,
                         "point_like_prob_cut": cut, **selection_snr(regions, in_match_filter & (point_like_prob >= cut))})
results = pd.DataFrame(rows)
reference_snr = results[np.logical_and.reduce([results[key] == value for key, value in REFERENCE.items()])]["snr"].iloc[0]
results["relative_snr"] = results["snr"] / reference_snr
print(f"{len(results)} scenarios; reference SNR {reference_snr:.2f}")
results.sort_values("snr", ascending=False).head(10).round(2)

In [ ]:
def select(table, **conditions):
    """Rows of a results table matching every condition (column=value)."""
    return table[np.logical_and.reduce([table[key] == value for key, value in conditions.items()])]


# SNR against the POINT_LIKE_PROB cut: classification (colour) and isochrone model (line style), for some widths
POINT_LIKE_PROB_SCAN_WIDTHS = ["default", "color_spread (0.1, 0.1)"]
fig, axes = plt.subplots(1, len(POINT_LIKE_PROB_SCAN_WIDTHS), figsize=(5 * len(POINT_LIKE_PROB_SCAN_WIDTHS), 3.5),
                         sharey=True, layout="constrained")
for ax, width_name in zip(np.atleast_1d(axes), POINT_LIKE_PROB_SCAN_WIDTHS):
    for (classification, color), (isochrone_model, linestyle) in itertools.product(
            CLASSIFICATION_COLORS.items(), ISOCHRONE_MODEL_LINESTYLES.items()):
        curve = select(results, classification=classification, isochrone_model=isochrone_model, width=width_name)
        ax.plot(curve["point_like_prob_cut"], curve["snr"], color=color, ls=linestyle, marker=".",
                label=f"{classification}, {isochrone_model}")
    ax.axhline(reference_snr, color="0.6", lw=0.8)
    ax.set(title=escape_text(f"{track.track_name}, width {width_name}"), xlabel=escape_text("POINT_LIKE_PROB cut"),
           ylabel="SNR")
np.atleast_1d(axes)[-1].legend(fontsize="x-small")
plt.show()

In [ ]:
# SNR of the widths and isochrone models, at the POINT_LIKE_PROB cut of each classification
SCENARIO_POINT_LIKE_PROB_CUTS = {"morphology": 0.5, "gatto": 0.55}
ax = None
for (classification, color), (isochrone_model, marker) in itertools.product(
        CLASSIFICATION_COLORS.items(), ISOCHRONE_MODEL_MARKERS.items()):
    scenario = select(results, classification=classification, isochrone_model=isochrone_model,
                      point_like_prob_cut=SCENARIO_POINT_LIKE_PROB_CUTS[classification])
    fig, ax = plot_selection_comparison(scenario.set_index("width").loc[list(WIDTHS)], ax=ax, marker=marker,
                                        color=color, label=f"{classification}, {isochrone_model}")
ax.axvline(reference_snr, color="0.6", lw=0.8)
ax.set_title(escape_text(track.track_name))
ax.legend(fontsize="small", loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.show()

In [ ]:
# Best scenarios of each classification
best = results.sort_values("snr", ascending=False).groupby("classification").head(5)
best[["classification", "isochrone_model", "width", "point_like_prob_cut", "n_on", "n_background_on", "n_signal",
      "snr", "relative_snr"]].round(2)

## 6. Distance of the match filter

SNR of one scenario with the match filter at the distance of the track plus an offset: a check of the track distance.

In [ ]:
DM_OFFSETS = np.round(np.arange(-1.0, 1.01, 0.1), 2)
DM_SCAN = {"classification": "morphology", "isochrone_model": "Marigo2017", "width": "default", "point_like_prob_cut": 0.5}

stars, regions = fields[DM_SCAN["classification"]]
kept = stars["POINT_LIKE_PROB"].to_numpy() >= DM_SCAN["point_like_prob_cut"]
match_filter = {**MATCH_FILTER, **WIDTHS[DM_SCAN["width"]]}
dm_snr = [selection_snr(regions, track_match_filter_mask(stars, regions, ISOCHRONES[DM_SCAN["isochrone_model"]],
                                                         match_filter, dm_offset=offset) & kept)["snr"]
          for offset in DM_OFFSETS]
fig, ax = plt.subplots(figsize=(5, 3.5), layout="constrained")
ax.plot(DM_OFFSETS, dm_snr, marker=".")
ax.axvline(0.0, color="0.6", lw=0.8)
ax.set(xlabel="distance modulus offset from the track", ylabel="SNR", title=escape_text(track.track_name))
plt.show()